In [6]:
# Step 1: Load the dataset
import pandas as pd
import numpy as np


df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="latin-1")

raw_df = df.copy(deep=True)

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Dataset loaded successfully
Rows: 180519
Columns: 53


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [5]:
# Step 2: Display all column names
print("Total number of columns:", len(df.columns))
print("\nColumn names:")
for column in df.columns:
    print(column)



Total number of columns: 53

Column names:
Type
Days for shipping (real)
Days for shipment (scheduled)
Benefit per order
Sales per customer
Delivery Status
Late_delivery_risk
Category Id
Category Name
Customer City
Customer Country
Customer Email
Customer Fname
Customer Id
Customer Lname
Customer Password
Customer Segment
Customer State
Customer Street
Customer Zipcode
Department Id
Department Name
Latitude
Longitude
Market
Order City
Order Country
Order Customer Id
order date (DateOrders)
Order Id
Order Item Cardprod Id
Order Item Discount
Order Item Discount Rate
Order Item Id
Order Item Product Price
Order Item Profit Ratio
Order Item Quantity
Sales
Order Item Total
Order Profit Per Order
Order Region
Order State
Order Status
Order Zipcode
Product Card Id
Product Category Id
Product Description
Product Image
Product Name
Product Price
Product Status
shipping date (DateOrders)
Shipping Mode


In [7]:
# Step 3: Check data types and non-null values

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [8]:
# Step 4: Check invalid order quantities

invalid_quantity = df[
    pd.to_numeric(df["Order Item Quantity"], errors="coerce") <= 0
]

print("Invalid quantity rows:", len(invalid_quantity))
display(invalid_quantity.head())


Invalid quantity rows: 0


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode


In [9]:
# Step 5: Check negative product prices

price = pd.to_numeric(df["Product Price"], errors="coerce")

invalid_price = df[price < 0]

print("Negative product price rows:", len(invalid_price))
display(invalid_price.head())


Negative product price rows: 0


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode


In [11]:

# Step 6: Check suspicious numeric values

numeric_rules = {
    "Order Item Quantity": lambda s: s < 0,
    "Sales": lambda s: s < 0,
    "Product Price": lambda s: s < 0,
    "Order Item Discount": lambda s: s < 0,
    "Days for shipping (real)": lambda s: s < 0,
    "Days for shipment (scheduled)": lambda s: s < 0
}

numeric_findings = []

for column, rule in numeric_rules.items():
    if column in df.columns:
        values = pd.to_numeric(df[column], errors="coerce")
        invalid_mask = rule(values).fillna(False)

        numeric_findings.append({
            "Issue": f"Suspicious numeric values in {column}",
            "Affected Records": int(invalid_mask.sum())
        })

numeric_report = pd.DataFrame(numeric_findings)
display(numeric_report)



,Issue,Affected Records
0,Suspicious numeric values in Order Item Quantity,0
1,Suspicious numeric values in Sales,0
2,Suspicious numeric values in Product Price,0
3,Suspicious numeric values in Order Item Discount,0
4,Suspicious numeric values in Days for shipping...,0
5,Suspicious numeric values in Days for shipment...,0


In [12]:
# Step 7: Check inconsistent text formats and category labels

text_columns = [
    "Category Name",
    "Department Name",
    "Order Region",
    "Delivery Status",
    "Type",
    "Customer Country",
    "Order Country"
]

format_findings = []
label_findings = []

for column in text_columns:
    if column in df.columns:
        values = df[column].dropna().astype(str)

        # Leading or trailing whitespace
        whitespace_count = int(
            values.ne(values.str.strip()).sum()
        )
        format_findings.append({
            "Column": column,
            "Issue": "Leading/trailing whitespace",
            "Affected Records": whitespace_count
        })

        # Labels that differ only by case or extra spaces
        normalized = values.str.strip().str.casefold()
        label_map = pd.DataFrame({
            "Original": values,
            "Normalized": normalized
        })

        variant_counts = (
            label_map.groupby("Normalized")["Original"]
            .nunique()
        )

        inconsistent_labels = variant_counts[variant_counts > 1]

        label_findings.append({
            "Column": column,
            "Groups with label variations": len(inconsistent_labels)
        })

print("Format findings:")
display(pd.DataFrame(format_findings))

print("Possible category-label variations:")
display(pd.DataFrame(label_findings))


Format findings:


,Column,Issue,Affected Records
0,Category Name,Leading/trailing whitespace,1475
1,Department Name,Leading/trailing whitespace,362
2,Order Region,Leading/trailing whitespace,17925
3,Delivery Status,Leading/trailing whitespace,0
4,Type,Leading/trailing whitespace,0
5,Customer Country,Leading/trailing whitespace,0
6,Order Country,Leading/trailing whitespace,0


Possible category-label variations:


,Column,Groups with label variations
0,Category Name,0
1,Department Name,0
2,Order Region,0
3,Delivery Status,0
4,Type,0
5,Customer Country,0
6,Order Country,0


In [14]:
# Step 8: Check invalid date values

date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

date_findings = []

for column in date_columns:
    if column in df.columns:
        original = df[column]
        parsed = pd.to_datetime(original, errors="coerce")

        non_empty = (
            original.notna()
            & original.astype(str).str.strip().ne("")
        )

        invalid_mask = non_empty & parsed.isna()

        date_findings.append({
            "Column": column,
            "Invalid date records": int(invalid_mask.sum()),
            "Blank records": int((~non_empty).sum())
        })

display(pd.DataFrame(date_findings))


,Column,Invalid date records,Blank records
0,order date (DateOrders),0,0
1,shipping date (DateOrders),0,0


In [15]:

# Step 9: Check whether shipping date is before order date

order_dates = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

shipping_dates = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

comparable = order_dates.notna() & shipping_dates.notna()

invalid_shipping_order = (
    comparable & (shipping_dates < order_dates)
)

print(
    "Shipping-before-order records:",
    int(invalid_shipping_order.sum())
)

display(
    df.loc[
        invalid_shipping_order,
        ["order date (DateOrders)", "shipping date (DateOrders)"]
    ].head()
)


Shipping-before-order records: 0


,order date (DateOrders),shipping date (DateOrders)


In [16]:
# Step 10: Save Day 2 Invalid-Value Analysis Report

report_rows = []

# Numeric checks
for item in numeric_findings:
    report_rows.append({
        "Category": "Numeric Validation",
        "Issue": item["Issue"],
        "Affected Records": item["Affected Records"]
    })

# Text format checks
for item in format_findings:
    report_rows.append({
        "Category": "Format Validation",
        "Issue": f'{item["Column"]}: {item["Issue"]}',
        "Affected Records": item["Affected Records"]
    })

# Category label review
for item in label_findings:
    report_rows.append({
        "Category": "Category Label Review",
        "Issue": f'{item["Column"]}: possible label variations',
        "Affected Records": item["Groups with label variations"]
    })

# Invalid date checks
for item in date_findings:
    report_rows.append({
        "Category": "Date Validation",
        "Issue": f'{item["Column"]}: invalid dates',
        "Affected Records": item["Invalid date records"]
    })

# Shipping date before order date
report_rows.append({
    "Category": "Date Validation",
    "Issue": "Shipping date earlier than order date",
    "Affected Records": int(invalid_shipping_order.sum())
})

final_report = pd.DataFrame(report_rows)

with pd.ExcelWriter("Day2_Invalid_Value_Report.xlsx") as writer:
    final_report.to_excel(
        writer, sheet_name="Issue Summary", index=False
    )
    numeric_report.to_excel(
        writer, sheet_name="Numeric Checks", index=False
    )
    pd.DataFrame(format_findings).to_excel(
        writer, sheet_name="Format Checks", index=False
    )
    pd.DataFrame(label_findings).to_excel(
        writer, sheet_name="Label Review", index=False
    )
    pd.DataFrame(date_findings).to_excel(
        writer, sheet_name="Date Checks", index=False
    )

print("Day 2 report saved successfully!")
print("File: Day2_Invalid_Value_Report.xlsx")

display(final_report)


Day 2 report saved successfully!
File: Day2_Invalid_Value_Report.xlsx


,Category,Issue,Affected Records
0,Numeric Validation,Suspicious numeric values in Order Item Quantity,0
1,Numeric Validation,Suspicious numeric values in Sales,0
2,Numeric Validation,Suspicious numeric values in Product Price,0
3,Numeric Validation,Suspicious numeric values in Order Item Discount,0
4,Numeric Validation,Suspicious numeric values in Days for shipping...,0
5,Numeric Validation,Suspicious numeric values in Days for shipment...,0
6,Format Validation,Category Name: Leading/trailing whitespace,1475
7,Format Validation,Department Name: Leading/trailing whitespace,362
8,Format Validation,Order Region: Leading/trailing whitespace,17925
9,Format Validation,Delivery Status: Leading/trailing whitespace,0
